# Fact verifier verifier2 -- qwen3.5-4b (Colab)

Trains a LoRA adapter on `Qwen/Qwen3.5-4B` that labels every fact ckpt100 extracted from a
chunk: `stated` (with a quote from the chunk), `unsupported` (with a category) or
`previous_only`. Training data: the dpo4 Opus labels (`scripts/distill/verifier_data.py`).
Spec: `docs/superpowers/specs/2026-10-27-fact-verifier-design.md`.

**verifier2 is the spec's one recall follow-up (section 6).** verifier1 (ckpt200) caught
0.275 of the unsupported facts at 0.013 false drops (P2): a recall-only miss. verifier2
trains on the same data with each reply holding an unsupported fact repeated 3 times;
dev and test are unchanged, so the two runs compare directly.

**Selection:** the checkpoint with the lowest dev loss. Training stops by itself once the
last `DEV_PATIENCE` evals all sit above the best before them.

**Guards from the DPO notebook:** causal_conv1d kernel off for training, torch.compile off,
the gradient guard, the serving-prompt check and the loss-mask check.

**Use:** set the HF token secret (section 2), then **Runtime -> Run all**. A dead runtime
resumes from the last checkpoint on Drive, and the predictions resume where they stopped.
Hand back the adapter revision section 10 prints; the test predictions are uploaded with it
and scored locally (`scripts/distill/verifier_eval.py`).

## 1. Install packages

Same install as `distill_train.ipynb` section 1.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def pip_install(*args: str) -> None:
    """Run pip, print ALL of its output, and raise on failure (never hidden)."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout, flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"pip install failed (exit {proc.returncode}): {' '.join(args)}")


if importlib.util.find_spec("unsloth") is None:
    pip_install("--upgrade", "pip")
    pip_install("unsloth")
    pip_install("trl", "datasets", "huggingface_hub")

for pkg in ("unsloth", "trl", "transformers", "peft", "datasets", "huggingface_hub"):
    print(f"{pkg}=={importlib.metadata.version(pkg)}")
print("packages ready")

## 1a. causal_conv1d kernel OFF for training, torch.compile OFF

Verbatim from `distill_dpo.ipynb` section 1a (dpo2: the hub kernel's backward gave NaN gradients; dpo3: torch.compile hung for 2 h in step 1).

In [ ]:
import importlib.util
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

# torch.compile OFF (dpo3, 2026-10-23): it spent 2 h of CPU tracing the first training
# step's decoder layers, GPU at 0%, no step finished. Checked per call, so every
# torch.compile'd function from here on (Unsloth's layer compile included) runs eagerly.
torch._dynamo.config.disable = True
print("torch.compile: off (torch._dynamo.config.disable = True)")

CC1D_REPO, CC1D_REVISION = "kernels-community/causal-conv1d", "v1"
_QWEN_MOD = "transformers.models.qwen3_5.modeling_qwen3_5"


def load_causal_conv1d():
    tv = "".join(torch.__version__.split("+")[0].split(".")[:2])
    variant = f"torch{tv}-cxx11-cu{torch.version.cuda.replace('.', '')}-x86_64-linux"
    root = Path(snapshot_download(CC1D_REPO, revision=CC1D_REVISION,
                                  allow_patterns=[f"build/{variant}/*"]))
    pkg = root / "build" / variant
    if not (pkg / "__init__.py").exists():
        raise FileNotFoundError(f"{CC1D_REPO}@{CC1D_REVISION} has no {variant} build")
    spec = importlib.util.spec_from_file_location(
        "causal_conv1d", pkg / "__init__.py", submodule_search_locations=[str(pkg)])
    mod = importlib.util.module_from_spec(spec)
    sys.modules["causal_conv1d"] = mod
    spec.loader.exec_module(mod)
    return mod, variant


def check_causal_conv1d(mod) -> None:
    # Compare against the reference implementations shipped in the same package.
    ref = mod.causal_conv1d_interface
    g = torch.Generator(device="cuda").manual_seed(0)
    x = torch.randn(2, 96, 333, device="cuda", dtype=torch.bfloat16, generator=g)
    w = torch.randn(96, 4, device="cuda", dtype=torch.bfloat16, generator=g)
    b = torch.randn(96, device="cuda", dtype=torch.bfloat16, generator=g)
    xk, xr = x.clone().requires_grad_(), x.float().clone().requires_grad_()
    out = mod.causal_conv1d_fn(xk, w, b, activation="silu")
    want = ref.causal_conv1d_ref(xr, w.float(), b.float(), activation="silu")
    assert torch.allclose(out.float(), want, atol=0.1, rtol=0.05), "forward mismatch"
    out.float().sum().backward()
    want.sum().backward()
    assert torch.allclose(xk.grad.float(), xr.grad, atol=0.1, rtol=0.05), "backward mismatch"
    state = torch.randn(2, 96, 4, device="cuda", dtype=torch.bfloat16, generator=g)
    step = torch.randn(2, 96, 1, device="cuda", dtype=torch.bfloat16, generator=g)
    s_k, s_r = state.clone(), state.float().clone()
    out = mod.causal_conv1d_update(step, s_k, w, b, activation="silu")
    want = ref.causal_conv1d_update_ref(step.float(), s_r, w.float(), b.float(), activation="silu")
    assert torch.allclose(out.float(), want, atol=0.1, rtol=0.05), "decode-step mismatch"
    assert torch.allclose(s_k.float(), s_r, atol=0.05), "decode-step state mismatch"


USE_HUB_CAUSAL_CONV1D = False   # dpo2: its BACKWARD gave NaN / ~5e5 gradient norms
CC1D_LOADED = False
if not USE_HUB_CAUSAL_CONV1D:
    print("causal_conv1d: hub kernel off (USE_HUB_CAUSAL_CONV1D = False) -- PyTorch path")
elif _QWEN_MOD in sys.modules:
    print("transformers' Qwen3.5 code is already imported, so a kernel registered now would "
          "be ignored -- Runtime -> Restart session, then Run all")
else:
    try:
        _cc1d, _variant = load_causal_conv1d()
        check_causal_conv1d(_cc1d)
        CC1D_LOADED = True
        print(f"causal_conv1d: {CC1D_REPO} {_variant} loaded and matches its reference")
    except Exception as e:
        for _k in [k for k in sys.modules if k.split(".")[0] == "causal_conv1d"]:
            del sys.modules[_k]
        print(f"causal_conv1d: hub kernel NOT used ({type(e).__name__}: {e}) -- "
              "training runs on the PyTorch path (correct, slower)")

## 2. HF token -- Colab secret or environment, never hardcoded

Set a Colab secret named `HF_TOKEN` (the key icon in the left sidebar) *or* export
`HF_TOKEN` in the environment before starting this runtime. This cell never prints the
token -- only whether one was found.

In [ ]:
import os


def get_hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok
        except Exception:
            pass  # secret not set -- fall through to the environment
    except ImportError:
        pass  # not running on Colab
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        raise RuntimeError(
            "no HF token found -- add a Colab secret named HF_TOKEN, or set the "
            "HF_TOKEN environment variable, before running this cell")
    return tok


HF_TOKEN = get_hf_token()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF token loaded and logged in (value never printed)")

## 3. Configuration

In [ ]:
from pathlib import Path

# --- dataset: verifier_data.py output (train / val = dev / test) ----------------------------
DATA_SOURCE = "hf_dataset"
HF_DATASET_REPO = "carev01/ft-opus-distill"   # private
HF_DATASET_REVISION = "verifier2"
DRIVE_DATA_DIR = "/content/drive/MyDrive/ft-opus-verifier2"   # only when DATA_SOURCE == "drive"

# --- Google Drive: checkpoints, final adapter, predictions (survive a runtime death) ---------
RUN_NAME = "opus-verifier2"
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/graph-rag-distill") / RUN_NAME
CKPT_DIR = DRIVE_RUN_DIR / "checkpoints"
FINAL_ADAPTER_DIR = DRIVE_RUN_DIR / "final-adapter"
FINAL_MARKER = FINAL_ADAPTER_DIR / "COMPLETE"   # written LAST: adapter fully saved

# --- model -------------------------------------------------------------------------------------
BASE_MODEL_ID = "Qwen/Qwen3.5-4B"   # the plain model, not ckpt100: no shared extraction habits
MAX_SEQ_LENGTH = 20480
LOAD_IN_4BIT = True

# --- LoRA (same shape as the SFT run) ----------------------------------------------------
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# --- training (SFT settings of distill_train.ipynb) --------------------------------------
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "cosine"
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16   # 32 on an L4 if 16 runs out of memory
EVAL_STEPS = 25            # also the save interval, so the selected checkpoint is on Drive
SAVE_TOTAL_LIMIT = None    # SFT v2's best checkpoint was rotated out; adapters are small
DEV_PATIENCE = 3           # stop once the last 3 evals all sit above the best before them
SEED = 20261027
GRAD_NORM_LIMIT = 1000.0   # training guard: stop on a larger norm from GRAD_NORM_FROM_STEP on
GRAD_NORM_FROM_STEP = 12

# --- predictions on test, and the hand-off ---------------------------------------------------
PRED_BATCH = 8
HF_ADAPTER_REPO = "carev01/qwen35-4b-graphrag-verifier"   # private

OUT_DIR = Path("/content/outputs/opus-verifier")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("config set")

## 3a. Mount Google Drive

Same as `distill_train.ipynb` section 3a.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
_ckpts = sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print(f"run dir: {DRIVE_RUN_DIR}")
print(f"checkpoints on Drive: {[p.name for p in _ckpts] or 'none'}")
print(f"final adapter on Drive (complete): {FINAL_MARKER.exists()}")

## 4. Get the dataset

In [ ]:
DATA_DIR = Path("/content/data/ft-opus-verifier2")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "drive":
    DATA_DIR = Path(DRIVE_DATA_DIR)
elif DATA_SOURCE == "hf_dataset":
    from huggingface_hub import hf_hub_download
    for fname in ("train.jsonl", "val.jsonl", "test.jsonl", "MANIFEST.json"):
        p = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                            filename=fname, revision=HF_DATASET_REVISION, token=HF_TOKEN)
        (DATA_DIR / fname).write_bytes(Path(p).read_bytes())
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

assert all((DATA_DIR / f).exists() for f in ("train.jsonl", "val.jsonl", "test.jsonl")), DATA_DIR
print(f"dataset ready at {DATA_DIR}")

## 5. Load the splits and check their shape (fail loudly)

In [ ]:
import json
from collections import Counter


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_records = read_jsonl(DATA_DIR / "train.jsonl")
val_records = read_jsonl(DATA_DIR / "val.jsonl")      # the dev split
test_records = read_jsonl(DATA_DIR / "test.jsonl")
for _name, _rows in (("train", train_records), ("dev", val_records), ("test", test_records)):
    for _r in _rows:
        assert [m["role"] for m in _r["messages"]] == ["system", "user", "assistant"], _r["meta"]
        _labels = json.loads(_r["messages"][2]["content"])["labels"]
        assert len(_labels) == len(_r["meta"]["final"]), _r["meta"]["call_id"]
    _finals = Counter(f for r in _rows for f in r["meta"]["final"])
    print(f"{_name}: {len(_rows)} replies, facts {dict(sorted(_finals.items()))}")
_groups = {s: {r["meta"]["group"] for r in rows}
           for s, rows in (("train", train_records), ("dev", val_records), ("test", test_records))}
assert not (_groups["train"] & _groups["test"]) and not (_groups["dev"] & _groups["test"]), (
    "an article is on both sides of the test split")

## 6. Load the base model and attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
    token=HF_TOKEN,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("base model loaded, LoRA attached")
import transformers.models.qwen3_5.modeling_qwen3_5 as _mq


def calls_package(fn, package: str) -> bool:
    # Follow transformers' kernel wrappers down to the implementation they call.
    seen, stack = set(), [fn]
    while stack:
        f = stack.pop()
        if f is None or id(f) in seen or not callable(f):
            continue
        seen.add(id(f))
        if str(getattr(f, "__module__", "")).split(".")[0] == package:
            return True
        stack.append(getattr(f, "__wrapped__", None))
        for c in getattr(f, "__closure__", None) or ():
            try:
                stack.append(c.cell_contents)
            except ValueError:
                pass
    return False


_uses_cc1d = calls_package(_mq.causal_conv1d_fn, "causal_conv1d")
print(f"model calls a causal_conv1d package: {_uses_cc1d}")
assert USE_HUB_CAUSAL_CONV1D or not _uses_cc1d, (
    "the hub causal_conv1d kernel is off, yet the model calls a causal_conv1d package -- "
    "its backward pass broke dpo2; restart the session without it")

## 7. Build the chat-formatted text (thinking-off serving prompt + answer)

In [ ]:
from datasets import Dataset

END_OF_TURN = "<|im_end|>\n"


def build_training_text(messages: list[dict]) -> tuple[str, int, str]:
    """Mirrors scripts/distill/chat_format.py:build_training_text. Returns
    (text, mask_start, path) -- see section 0 and that module's docstring."""
    assert messages[-1]["role"] == "assistant", messages
    answer = messages[-1]["content"]
    prefix = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True,
        enable_thinking=False)
    constructed = prefix + answer + END_OF_TURN
    plain = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    path = "plain" if plain == constructed else "generation_prompt"
    return constructed, len(prefix), path


def to_text(example: dict) -> dict:
    text, mask_start, path = build_training_text(example["messages"])
    return {"text": text, "mask_start": mask_start, "path": path}


train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_records]).map(
    to_text, remove_columns=["messages"])
val_ds = Dataset.from_list([{"messages": r["messages"]} for r in val_records]).map(
    to_text, remove_columns=["messages"])

path_counts = Counter(train_ds["path"])
print(f"construction path across train: {dict(path_counts)}")
print(train_ds[0]["text"][:800])

train_ds = train_ds.remove_columns(["mask_start", "path"])
val_ds = val_ds.remove_columns(["mask_start", "path"])

## 8. Verify: training text == serving prompt + answer + end-of-turn (fail loudly)

In [ ]:
_probe = train_records[0]
_serving = tokenizer.apply_chat_template(
    _probe["messages"][:-1], tokenize=False, add_generation_prompt=True,
    enable_thinking=False)
_answer = _probe["messages"][-1]["content"]
_expected = _serving + _answer + END_OF_TURN
_built_text, _built_mask_start, _built_path = build_training_text(_probe["messages"])

if _built_text != _expected:
    _divergence = next(
        (i for i in range(min(len(_built_text), len(_expected)))
         if _built_text[i] != _expected[i]), None)
    raise AssertionError(
        "training text does not equal serving prompt + answer + end-of-turn marker -- "
        "training and serving would NOT target the same completion. "
        f"path={_built_path!r}, first divergence at char {_divergence!r}. "
        f"built[:200]={_built_text[:200]!r} expected[:200]={_expected[:200]!r}")
assert _built_mask_start == len(_serving), (
    f"mask_start {_built_mask_start} != len(serving prompt) {len(_serving)}")

print(f"OK: training text == serving prompt + answer + end-of-turn (path={_built_path})")
print("serving prompt tail:", repr(_serving[-70:]))

## 9. Train -- completion-only loss, resumable

The gradient guard prints the first `GRAD_NORM_FROM_STEP` gradient norms, stops on a NaN or
inf gradient before the optimiser step, and on a norm above `GRAD_NORM_LIMIT` later.
`DevLossStop` ends training once the last `DEV_PATIENCE` evals all sit above the best one
before them.

In [ ]:
import dataclasses
import inspect

from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

"""Verifier checkpoint selection (spec 2026-10-27-fact-verifier-design.md section 4): the
lowest dev loss, and an early stop once the last `patience` evals all sit above the best
before them (SFT v2 overfitted after about half an epoch). build_verifier_nb.py embeds this
module verbatim in the notebook: the unit tests and Colab run the same code."""


def dev_losses(log_history):
    """(step, eval_loss) per eval, sorted, one per step (a resumed run repeats some)."""
    by_step = {int(h["step"]): float(h["eval_loss"]) for h in log_history if "eval_loss" in h}
    return sorted(by_step.items())


def should_stop(losses, patience):
    return len(losses) > patience and min(losses[-patience:]) > min(losses[:-patience])


def select_dev_step(log_history):
    """The step with the lowest dev loss (the earlier on a tie), or None with no eval."""
    series = dev_losses(log_history)
    return min(series, key=lambda sv: (sv[1], sv[0]))[0] if series else None


class GradGuard(TrainerCallback):
    # Stop on the first non-finite gradient, BEFORE the optimiser step (dpo2 skipped 40 of
    # 72 steps on NaN without a word).
    checked = 0

    def on_pre_optimizer_step(self, args, state, control, model=None, **kw):
        if not self.checked:
            print("  grad guard: inspecting gradients before every optimiser step", flush=True)
        self.checked += 1
        bad = [n for n, p in model.named_parameters()
               if p.grad is not None and not torch.isfinite(p.grad).all()]
        if bad:
            raise RuntimeError(f"step {state.global_step + 1}: non-finite gradients in "
                               f"{len(bad)} tensors (first: {bad[0]}) -- stopping")

    def on_log(self, args, state, control, logs=None, **kw):
        g = (logs or {}).get("grad_norm")
        if g is None:
            return
        if not self.checked:
            raise RuntimeError("grad guard never ran before an optimiser step -- the NaN check "
                               "is not live")
        if state.global_step <= GRAD_NORM_FROM_STEP:
            print(f"  step {state.global_step}: grad_norm {g:.4g}", flush=True)
        elif g > GRAD_NORM_LIMIT:
            raise RuntimeError(f"step {state.global_step}: grad_norm {g:.4g} > "
                               f"{GRAD_NORM_LIMIT} -- stopping")


class DevLossStop(TrainerCallback):
    # Stateless (reads state.log_history), so a resumed run keeps its history.
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        losses = [v for _, v in dev_losses(state.log_history)]
        if should_stop(losses, DEV_PATIENCE):
            print(f"  step {state.global_step}: dev loss above its best for {DEV_PATIENCE} "
                  "evals -- stopping", flush=True)
            control.should_training_stop = True
        return control


_probe_prefix = tokenizer.apply_chat_template(
    [{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True,
    enable_thinking=False)
RESPONSE_PART = _probe_prefix.rsplit(END_OF_TURN, 1)[-1]
INSTRUCTION_PART = "<|im_start|>user\n"
_fields = {f.name for f in dataclasses.fields(SFTConfig)}
_extra = {"max_seq_length" if "max_seq_length" in _fields else "max_length": MAX_SEQ_LENGTH,
          "warmup_ratio" if "warmup_ratio" in _fields else "warmup_steps": WARMUP_RATIO}
print(f"SFTConfig extras: {_extra}")

sft_config = SFTConfig(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim="adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=False,   # selection is section 10's, from log_history
    dataset_text_field="text",
    packing=False,                  # train_on_responses_only masks per example
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    **_extra,
)
_tok_kw = ("processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
           else "tokenizer")
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=sft_config,
                     callbacks=[GradGuard(), DevLossStop()], **{_tok_kw: tokenizer})
trainer = train_on_responses_only(trainer, instruction_part=INSTRUCTION_PART,
                                  response_part=RESPONSE_PART)

# --- mask check: the loss covers exactly answer + "<|im_end|>" ---------------------
_labels = list(trainer.train_dataset[0]["labels"])
_unmasked = [t for t in _labels if t != -100]
_decoded = tokenizer.decode(_unmasked, skip_special_tokens=False)
_expected = train_records[0]["messages"][-1]["content"] + "<|im_end|>"
# Current TRL appends the tokenizer's EOS (<|im_end|>) to text that already ends with the
# template's "<|im_end|>\n": harmless, generation stops at the first <|im_end|>.
assert _decoded in (_expected, _expected + "\n", _expected + "\n<|im_end|>"), (
    f"train_on_responses_only left the wrong tokens unmasked: {_decoded[:200]!r}")
if _decoded.endswith("\n<|im_end|>"):
    print("note: TRL appends a second <|im_end|> after the template's end of turn (harmless)")
print(f"OK: unmasked labels == answer + '<|im_end|>' ({len(_unmasked)} of {len(_labels)} tokens)")

In [ ]:
import json as _json

_CKPT_FILES = ("trainer_state.json", "optimizer.pt", "scheduler.pt")


def last_complete_checkpoint(ckpt_dir: Path) -> str | None:
    for c in sorted(ckpt_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]),
                    reverse=True):
        if all((c / f).exists() for f in _CKPT_FILES) and any(c.glob("adapter_model.*")):
            return str(c)
        print(f"skipping incomplete {c.name}")
    return None


if FINAL_MARKER.exists():
    print(f"final adapter already on Drive ({FINAL_ADAPTER_DIR}) -- skipping to the selection "
          "(section 10). Delete that folder to train again.")
else:
    _last = last_complete_checkpoint(CKPT_DIR)
    print(f"resuming from {_last}" if _last else "no checkpoint on Drive -- starting fresh")
    train_result = trainer.train(resume_from_checkpoint=_last)
    model.save_pretrained(str(FINAL_ADAPTER_DIR))
    tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
    (DRIVE_RUN_DIR / "train_result.json").write_text(_json.dumps(train_result.metrics, indent=2))
    (DRIVE_RUN_DIR / "log_history.json").write_text(
        _json.dumps(trainer.state.log_history, indent=2))
    FINAL_MARKER.write_text("adapter + tokenizer fully saved\n")

## 10. Select the checkpoint, predict the test split, hand off

Picks the lowest dev loss from `log_history.json` and loads that checkpoint into a FRESH
model (the training model still holds the final-step adapter). Before the test split it
checks itself on dev records: replies must parse, and batched generation must give the
same verdicts as batch 1 (left padding on this hybrid-attention model), else predictions
run at batch 1. Then it writes greedy, thinking-off replies for every test record to Drive
(resumable: done call ids are skipped, a half-written last line is dropped) and uploads
the adapter, the log and the predictions to the PRIVATE repo `HF_ADAPTER_REPO`, revision
`verifier2-ckpt<step>`.

In [ ]:
import gc
import json as _json
import os

from huggingface_hub import HfApi

_hist = _json.loads((DRIVE_RUN_DIR / "log_history.json").read_text())
for _s, _v in dev_losses(_hist):
    print(f"eval @ {_s:>4}: dev loss {_v:.4f}")
SELECTED_STEP = select_dev_step(_hist)
assert SELECTED_STEP is not None, "no eval in the log history -- nothing to select"
_ckpt = CKPT_DIR / f"checkpoint-{SELECTED_STEP}"
assert (_ckpt / "adapter_model.safetensors").exists(), f"{_ckpt} has no adapter"
ADAPTER_REVISION = f"verifier2-ckpt{SELECTED_STEP}"

# A fresh model with the selected checkpoint's adapter (distill_train's reload pattern).
del trainer
del model
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(_ckpt), max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT,
    dtype=None, token=HF_TOKEN)
FastLanguageModel.for_inference(model)
TEXT_TOK = getattr(tokenizer, "tokenizer", tokenizer)
TEXT_TOK.padding_side = "left"
if TEXT_TOK.pad_token is None:
    TEXT_TOK.pad_token = TEXT_TOK.eos_token
END_ID = TEXT_TOK.convert_tokens_to_ids("<|im_end|>")
print(f"loaded {_ckpt.name} into a fresh model")


def serving_prompt(r: dict) -> str:
    return tokenizer.apply_chat_template(r["messages"][:-1], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)


def generate(batch: list[dict]) -> list[str]:
    enc = TEXT_TOK([serving_prompt(r) for r in batch], return_tensors="pt", padding=True,
                   add_special_tokens=False).to("cuda")
    cap = 64 + 160 * max(len(r["meta"]["final"]) for r in batch)   # facts, not answers
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=cap, do_sample=False,
                             eos_token_id=END_ID, pad_token_id=TEXT_TOK.pad_token_id)
    return [TEXT_TOK.decode(ids, skip_special_tokens=True)
            for ids in out[:, enc["input_ids"].shape[1]:]]


def verdicts(text: str, n: int):
    try:
        labels = _json.loads(text)["labels"]
        by_n = {lab["n"]: lab["verdict"] for lab in labels}
    except (ValueError, KeyError, TypeError):
        return None
    return tuple(by_n.get(k) for k in range(1, n + 1)) if len(by_n) == n == len(labels) else None


# --- self-check on dev: parse rate, agreement with the targets, batched == batch 1 ---------
_probe = sorted(val_records, key=lambda r: len(serving_prompt(r)))[:8]
_single = [generate([r])[0] for r in _probe]
_batched = [t for i in range(0, len(_probe), PRED_BATCH) for t in generate(_probe[i:i + PRED_BATCH])]
_n_facts = [len(r["meta"]["final"]) for r in _probe]
_v1 = [verdicts(t, n) for t, n in zip(_single, _n_facts)]
_vb = [verdicts(t, n) for t, n in zip(_batched, _n_facts)]
_want = [verdicts(r["messages"][2]["content"], n) for r, n in zip(_probe, _n_facts)]
_parsed = sum(v is not None for v in _v1)
_match = sum(a == b for v, w in zip(_v1, _want) if v is not None for a, b in zip(v, w))
_same = sum((a == b) if a is not None else (s == t)
            for a, b, s, t in zip(_v1, _vb, _single, _batched))
print(f"dev agreement: {_parsed}/{len(_probe)} replies parse at batch 1; verdicts match the "
      f"targets on {_match}/{sum(n for v, n in zip(_v1, _n_facts) if v is not None)} facts; "
      f"batched == batch 1 on {_same}/{len(_probe)}")
assert _parsed > 0, "no dev reply parses -- adapter or prompt is wrong; not predicting the test split"
if _same < len(_probe):
    print(f"WARNING: batch {PRED_BATCH} changes the verdicts -- predicting at batch 1")
    PRED_BATCH = 1


def drop_partial_tail(path: Path) -> None:
    # A runtime that died mid-write leaves a half line; appending to it would corrupt the file.
    if not path.exists():
        return
    lines = [x for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]
    good = []
    for x in lines:
        try:
            row = _json.loads(x)
        except ValueError:
            continue
        if isinstance(row, dict) and "call_id" in row and "reply" in row:
            good.append(x)
    if len(good) != len(lines):
        print(f"dropping {len(lines) - len(good)} unusable line(s) from {path.name}")
        path.write_text("".join(g + "\n" for g in good), encoding="utf-8")


PRED_PATH = DRIVE_RUN_DIR / f"test_predictions-ckpt{SELECTED_STEP}.jsonl"
drop_partial_tail(PRED_PATH)
_done = set()
if PRED_PATH.exists():
    _done = {_json.loads(x)["call_id"] for x in PRED_PATH.read_text().splitlines() if x.strip()}
_todo = sorted((r for r in test_records if r["meta"]["call_id"] not in _done),
               key=lambda r: len(serving_prompt(r)))
print(f"{len(_done)} predictions already on Drive, {len(_todo)} to go (batch {PRED_BATCH})")
# One open/append/close per batch: Colab's Drive mount uploads on close, so a file held
# open for the whole loop lost ~650 written rows when a runtime died (2026-10-27).
for _i in range(0, len(_todo), PRED_BATCH):
    _batch = _todo[_i:_i + PRED_BATCH]
    _rows = [_json.dumps({"call_id": _r["meta"]["call_id"], "reply": _text,
                          "step": SELECTED_STEP}, ensure_ascii=False) + "\n"
             for _r, _text in zip(_batch, generate(_batch))]
    with PRED_PATH.open("a", encoding="utf-8") as _fh:
        _fh.write("".join(_rows))
        _fh.flush()
        os.fsync(_fh.fileno())
    print(f"  {min(_i + PRED_BATCH, len(_todo))}/{len(_todo)}", flush=True)
_n = len({_json.loads(x)["call_id"] for x in PRED_PATH.read_text().splitlines() if x.strip()})
assert _n == len(test_records), f"{_n} predictions for {len(test_records)} test records"

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_ADAPTER_REPO, repo_type="model", private=True, exist_ok=True)
assert api.model_info(HF_ADAPTER_REPO).private, f"{HF_ADAPTER_REPO} is not private -- refusing"
api.create_branch(HF_ADAPTER_REPO, repo_type="model", branch=ADAPTER_REVISION, exist_ok=True)
for _src, _dst in ((_ckpt / "adapter_model.safetensors", "adapter_model.safetensors"),
                   (_ckpt / "adapter_config.json", "adapter_config.json"),
                   (DRIVE_RUN_DIR / "log_history.json", "log_history.json"),
                   (PRED_PATH, "test_predictions.jsonl")):
    api.upload_file(path_or_fileobj=str(_src), path_in_repo=_dst, repo_id=HF_ADAPTER_REPO,
                    repo_type="model", revision=ADAPTER_REVISION)
print(f"DONE: {HF_ADAPTER_REPO}@{ADAPTER_REVISION} -- hand this name back")

## 11. Next -- outside this notebook

Hand back the adapter revision. Scoring is local and free:
`uv run --extra dev python -m scripts.distill.verifier_eval --data <dataset dir>
--predictions <test_predictions.jsonl> --out <report.json>` applies the pre-registered bar
(spec section 6). Only on a pass does the production hook get its own spec.